In [ ]:
%run Training.ipynb
print("All done")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_A

**Upload model artifacts to S3**

In [ ]:
import boto3
import json
import joblib

s3 = boto3.client("s3")

model_key = f"{PREFIX}models/champion/credit_risk_model.joblib"
metrics_key = f"{PREFIX}models/champion/metrics.json"
schema_key = f"{PREFIX}models/champion/feature_schema.json"

# Upload model
s3.upload_file(
    "credit_risk_model.joblib",
    BUCKET,
    model_key
)

# Save metrics without the pipeline object
metrics = {
    name: {
        "roc_auc": result["roc_auc"],
        "average_precision": result["average_precision"],
        "confusion_matrix": result["confusion_matrix"],
        "classification_report": result["classification_report"]
    }
    for name, result in results.items()
}

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

s3.upload_file(
    "metrics.json",
    BUCKET,
    metrics_key
)

feature_schema = {
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target": target_col,
    "best_model": best_model_name
}

with open("feature_schema.json", "w") as f:
    json.dump(feature_schema, f, indent=2)

s3.upload_file(
    "feature_schema.json",
    BUCKET,
    schema_key
)